In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import logging
import h5py
import matplotlib.pyplot as plt  # type: ignore
import numpy as np
import healpy as hp
from itertools import product

from mlpng.utils import (
    setup_logging,
    plot_cl_alm,
    plot_predictions,
    plot_histogram,
    plot_elsner_comp,
    save_data,
    remove_mono_dipole,
    # pol_str,
    make_alm_plots,
)
from mlpng.generator import Generator
from mlpng.utils.utils import print_errors

# Setup logging for notebook
setup_logging("mlpng.notebook", level=logging.DEBUG)
logger = logging.getLogger("mlpng.notebook")

mpi_comm = None

## Setup

In [ ]:
generator = Generator(
    [
        "settings/n32.json",
        "--nsims",
        "100",
        "--narray",
        "1",
        "--fnl_range",
        "-1000",
        "1000",
        "--pols",
        "T",
        "--phi_scale",
        "30.0",
        "--shape",
        "local",
        "--no-noise",
        "--mc_steps",
        "300",
        "--force_generation",
        # "--nside",
        # "128",
        # "--lmax",
        # "383",
    ]
)

generator.check_existing_data_file()

In [ ]:
generator.run()
plt.show()

In [ ]:
stop  # 9.556421 #9.96
# 4.920115

In [ ]:
pol_idxs = generator.pol_idxs()

alm_l = generator.generate_alm()
alm_nl = generator.generate_alm_nl(alm_l)
# alm_nl = generator.generate_alm_nl_shape(alm_l, "local")
fnls = generator.rng.uniform(
    generator.fnl_min, generator.fnl_max, (generator.total_sims, 1, 1)
)

# Add the duplicate dimension to the alms
alm_l = alm_l[:, pol_idxs]

# finally combine into the full alms and remove the monopole and dipole terms
alms = alm_l + fnls * alm_nl

In [ ]:
# logger.info("Starting lensing")
# alm_lensed = generator.lens_alms(alms)

# maps_lensed = np.zeros(generator.map_shape, dtype=generator.r_dtype)
# for sim, dup in product(range(generator.nsims), range(generator.ndups)):
#     rmap = hp.alm2map(alm_lensed[sim], nside=generator.nside, pol=generator.use_pols)
#     maps_lensed[sim, dup] = hp.reorder(rmap, r2n=True)

## Alm

In [ ]:
print(alm_l.shape, alm_nl.shape, alms.shape)
plot_cl_alm(
    generator,
    alm_l[0],
    title="linear alms",
    # labels=plt_labels,
    plot_camb=True,
    plot_noise=False,
    plot_full_camb=False,
    show=True,
)

ells = np.arange(generator.nell)
scale = (ells * (ells + 1)) / (2 * np.pi)

plot_cl_alm(
    generator,
    alm_nl[0],
    title="non-linear alms",
    plot_camb=False,
    plot_noise=False,
    plot_full_camb=False,
    show=True,
    scale=True,
)

In [ ]:
plot_cl_alm(
    generator,
    alms[0],
    title="alms final",
    plot_camb=True,
    plot_noise=False,
    plot_full_camb=True,
    show=True,
)

In [ ]:
sim = generator.rng.integers(generator.nsims)
eidx = generator.rng.integers(1, 1001)
plot_elsner_comp(
    generator,
    alm_l[sim],
    alm_nl[sim],
    index=eidx,
    show=True,
    plot_func=plt.semilogy,
)
plot_elsner_comp(
    generator,
    alm_l[sim],
    alm_nl[sim],
    index=eidx,
    show=True,
    plot_func=plt.plot,
)
plot_elsner_comp(
    generator,
    alm_l[sim],
    alm_nl[sim],
    index=eidx,
    show=True,
    plot_func=plt.loglog,
)

In [ ]:
alm_l_avg = np.mean(alm_l, axis=0)
alm_ng_avg = np.mean(alm_nl, axis=0)

plot_elsner_comp(
    generator,
    alm_l_avg,
    alm_ng_avg,
    average=generator.nsims,
    title="avged comp",
    show=True,
    plot_func=plt.semilogy,
)

## Estimator

In [ ]:
mpi_size = 1

# the KSW code requires the total_sims to be >= mpi_size
# best usage would have total_sims % mpi_size == 0, but not required
assert (
    generator.num_estimates >= mpi_size
), "total_sims < mpi_size, lower ntasks or increase sims"

logger.info(
    "Computing %s estimates in %.2f batches",
    generator.num_estimates,
    generator.num_estimates / mpi_size,
)
if generator.num_estimates % mpi_size != 0:
    logger.warning(
        "num_estimates is not divisible by mpi_size, "
        "this will lead to uneven workloads."
    )

# The default theta_batch size is 25, which is really small, we want to increase it
theta_batch = int(np.floor(1.5 * generator.lmax + 1)) // mpi_size
logger.debug("Using theta_batch %s", theta_batch)


# alm_steps = generator.generate_alm(nsims=generator.mc_steps)

logger.info("Initializing KSW with %s steps", generator.mc_steps)
ksw = generator.get_ksw("local", step_alms=alm_l)
fisher = ksw.compute_fisher()
print(f"Fisher: {fisher}, standard deviation: {1 / np.sqrt(fisher)}")

In [ ]:
estimates, cubic, linear, fishers = ksw.compute_estimate_batch(
    lambda idx: generator.icov_func(alms[idx]),
    range(generator.nsims),
    # theta_batch=int(np.floor(1.5 * generator.lmax + 1)) // generator.slurm.n_cpus,
    fisher=fisher,
    # lin_term=0,
)

# print(cubic)
# print(linear)
# print(fishers)

fnls_flat = fnls.flatten()
print_errors(fnls_flat, estimates, fisher)
plot_predictions(fnls_flat, estimates, sigma=1 / np.sqrt(fisher), show=True)
plot_histogram(fnls_flat, estimates, show=True)

## Lensing

In [ ]:
alm_lensed = generator.lens_alms(alms)

In [ ]:
sim = 0
for i, pol in enumerate(generator.pol_idxs()):
    pstr = generator.pols[pol]

    ylabel = r"$\ell(\ell+1)/2\pi\;C_{\ell}" + f"^{pstr}$"
    plot_cl_alm(
        generator,
        alm_lensed[sim, i],
        # save_file=filebase,
        ylabel=ylabel,
        plot_camb=True,
        plot_noise=False,
        plot_full_camb=True,
        show=True,
        lmin=2,
    )

In [ ]:
lens_estimates, _, _, _ = ksw.compute_estimate_batch(
    lambda a: generator.icov_func(alm_lensed[a, generator.pol_idxs(pretrimmed=True)]),
    range(generator.nsims),
    theta_batch=int(np.floor(1.5 * generator.lmax + 1)) // generator.slurm.n_cpus,
    fisher=fisher,
    lin_term=0,
)

In [ ]:
print(
    "fnl shape", fnls.shape, "estimates shape", lens_estimates.shape, "fisher", fisher
)
fnls_flat = fnls[:, 0, 0]
print_errors(fnls_flat, lens_estimates, fisher)
plot_predictions(fnls_flat, lens_estimates, sigma=1 / np.sqrt(fisher), show=True)
plot_histogram(fnls_flat, lens_estimates, show=True)